#  Intermediate Fusion Pipeline: CoAtNet (Video) + Whisper (Audio)
## FakeAVCeleb v1.2 — Deepfake Detection via Feature-Level Fusion

**Methodology:**
1. **CoAtNet** (frozen, pretrained) → extract frame-level embeddings → Statistical Pooling (mean + max + std)
2. **Whisper** (frozen) → extract audio embeddings → Mean Pooling
3. **Concatenate** video + audio vectors → single fused representation
4. **XGBoost** classifier on the fused features

**Labels:** RealVideo-RealAudio → 0 (Bonafide) | All others → 1 (Spoof)

## 1. Environment Setup & Imports

In [ ]:
import os
import gc
import io
import json
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import librosa
import cv2
import subprocess
import soundfile as sf
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    precision_recall_fscore_support, average_precision_score
)

warnings.filterwarnings('ignore')
cv2.setNumThreads(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
GPU: Tesla T4


## 2. Configuration & Paths

In [ ]:
DATASET_ROOT = '/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2'
META_CSV     = os.path.join(DATASET_ROOT, 'meta_data.csv')
FRAMES_ROOT  = os.path.join(os.path.dirname(DATASET_ROOT), 'frames')

COATNET_WEIGHTS = '/artifacts_v2/best_coatnet_5ch.pth'

OUTPUT_DIR = '/content/drive/MyDrive/Work/DeepFakeg/fusion_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
NUM_FRAMES      = 5        # frames to sample per video
IMG_SIZE        = 224      # CoAtNet input resolution
WHISPER_MODEL   = 'openai/whisper-base'   # Whisper variant
WHISPER_DIM     = 512      # Whisper-base encoder hidden dim
TEST_RATIO      = 0.2
RANDOM_SEED     = 42

print('Configuration loaded.')
print(f'  Dataset root  : {DATASET_ROOT}')
print(f'  Metadata CSV  : {META_CSV}')
print(f'  Frames root   : {FRAMES_ROOT}')
print(f'  CoAtNet weights: {COATNET_WEIGHTS}')
print(f'  Output dir    : {OUTPUT_DIR}')

## 3. Dataset Pipeline — FakeAVCeleb Metadata

We rely **exclusively** on `meta_data.csv` as the single source of truth.
- **Label 0 (Bonafide):** `RealVideo-RealAudio`
- **Label 1 (Spoof):** `FakeVideo-FakeAudio`, `FakeVideo-RealAudio`, `RealVideo-FakeAudio`

In [ ]:
# 3.1 Load and parse metadata

meta_df = pd.read_csv(META_CSV)
print(f'Total entries in meta_data.csv: {len(meta_df)}')
print(f'Columns: {list(meta_df.columns)}')
print(f'\nSample rows:')
print(meta_df.head())

print(f'\nType distribution:')
print(meta_df['type'].value_counts())

print(f'\nMethod distribution:')
print(meta_df['method'].value_counts())

print(f'\nRace distribution:')
print(meta_df['race'].value_counts())

print(f'\nGender distribution:')
print(meta_df['gender'].value_counts())

Total entries in meta_data.csv: 21566
Columns: ['source', 'target1', 'target2', 'method', 'category', 'type', 'race', 'gender', 'path', 'Unnamed: 9']

Sample rows:
    source target1 target2 method category                 type     race  \
0  id00076       -       -   real        A  RealVideo-RealAudio  African   
1  id00166       -       -   real        A  RealVideo-RealAudio  African   
2  id00173       -       -   real        A  RealVideo-RealAudio  African   
3  id00366       -       -   real        A  RealVideo-RealAudio  African   
4  id00391       -       -   real        A  RealVideo-RealAudio  African   

  gender       path                                         Unnamed: 9  
0    men  00109.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
1    men  00010.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
2    men  00118.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
3    men  00118.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
4    men  0005

In [ ]:
# 3.2 Assign binary labels and construct absolute video paths
BONAFIDE_CLASS = 'RealVideo-RealAudio'

meta_df['label'] = meta_df['type'].apply(
    lambda x: 0 if x == BONAFIDE_CLASS else 1
)

# Construct full video path from metadata columns
def build_video_path(row):
    """Build absolute path: DATASET_ROOT/type/race/gender/source/filename."""
    return os.path.join(
        DATASET_ROOT,
        str(row['type']),
        str(row['race']),
        str(row['gender']),
        str(row['source']),
        str(row['path'])
    )

meta_df['abs_video_path'] = meta_df.apply(build_video_path, axis=1)

# Verify paths exist on disk
exists_mask = meta_df['abs_video_path'].apply(os.path.exists)
exists_count = exists_mask.sum()
print(f'Videos found on disk: {exists_count} / {len(meta_df)}')

# Show some examples
print(f'\nSample paths:')
for _, row in meta_df.head(3).iterrows():
    p = row['abs_video_path']
    e = 'EXISTS' if os.path.exists(p) else 'MISSING'
    print(f'  [{e}] {p}')

# Filter to only existing videos
if exists_count < len(meta_df):
    missing_count = len(meta_df) - exists_count
    print(f'\nWARNING: {missing_count} videos not found. Filtering to existing only.')
    meta_df = meta_df[exists_mask].reset_index(drop=True)

print(f'\nLabel distribution (after filtering):')
print(f'  Bonafide (0): {(meta_df["label"]==0).sum()}')
print(f'  Spoof    (1): {(meta_df["label"]==1).sum()}')

# Stratified Train/Test split
train_df, test_df = train_test_split(
    meta_df, test_size=TEST_RATIO, random_state=RANDOM_SEED,
    stratify=meta_df['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\nTrain set: {len(train_df)}  |  Test set: {len(test_df)}')
print(f'  Train bonafide: {(train_df["label"]==0).sum()}  spoof: {(train_df["label"]==1).sum()}')
print(f'  Test  bonafide: {(test_df["label"]==0).sum()}  spoof: {(test_df["label"]==1).sum()}')

# Save splits for reproducibility
train_df.to_csv(os.path.join(OUTPUT_DIR, 'train_split.csv'), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, 'test_split.csv'), index=False)
print('Splits saved.')

Videos found on disk: 21566 / 21566

Sample paths:
  [EXISTS] /kaggle/input/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00076/00109.mp4
  [EXISTS] /kaggle/input/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00166/00010.mp4
  [EXISTS] /kaggle/input/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00173/00118.mp4

Label distribution (after filtering):
  Bonafide (0): 500
  Spoof    (1): 21066

Train set: 17252  |  Test set: 4314
  Train bonafide: 400  spoof: 16852
  Test  bonafide: 100  spoof: 4214
Splits saved.


## 4. Feature Extractors — Frozen Models

Both models are loaded in **eval mode** with no gradient tracking.
- **CoAtNet**: Adapted 5-channel (RGB + ELA + FFT) from trained weights
- **Whisper**: `openai/whisper-base` encoder only

In [ ]:
# 4.1 ELA & FFT utilities 

def compute_ela(image_rgb, quality=90):
    """Error Level Analysis — highlights manipulation artifacts.
    Memory-safe: explicitly closes BytesIO buffer."""
    pil_img = Image.fromarray(image_rgb)
    buffer  = io.BytesIO()
    try:
        pil_img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        compressed = np.array(Image.open(buffer).convert('RGB')).astype(np.float32)
    finally:
        buffer.close()
        del pil_img
    original = image_rgb.astype(np.float32)
    ela_map = np.abs(original - compressed).mean(axis=2)
    del original, compressed
    ela_max = ela_map.max()
    if ela_max > 0:
        ela_map /= ela_max
    return ela_map.astype(np.float32)


def compute_fft_magnitude(image_rgb):
    """FFT magnitude spectrum — reveals periodic synthetic artifacts."""
    gray      = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    fft       = np.fft.fft2(gray)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.log1p(np.abs(fft_shift))
    del gray, fft, fft_shift
    mag_max   = magnitude.max()
    if mag_max > 0:
        magnitude /= mag_max
    return magnitude.astype(np.float32)


def prepare_5ch_tensor(image_rgb, size=224):
    """Convert RGB image to 5-channel normalized tensor for CoAtNet."""
    image_rgb = cv2.resize(image_rgb, (size, size))
    ela = compute_ela(image_rgb)
    fft = compute_fft_magnitude(image_rgb)
    ela_u8 = (ela * 255).astype(np.uint8)
    fft_u8 = (fft * 255).astype(np.uint8)
    del ela, fft
    ch5 = np.concatenate([
        image_rgb,
        ela_u8[..., np.newaxis],
        fft_u8[..., np.newaxis]
    ], axis=2)
    del ela_u8, fft_u8
    tensor = torch.from_numpy(ch5).permute(2, 0, 1).float() / 255.0
    del ch5
    mean = torch.tensor([0.485, 0.456, 0.406, 0.5, 0.5]).view(5, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225, 0.5, 0.5]).view(5, 1, 1)
    tensor = (tensor - mean) / std
    return tensor

print('ELA/FFT utilities loaded (memory-safe).')

ELA/FFT utilities loaded (memory-safe).


In [ ]:
# 4.2 Load CoAtNet as frozen feature extractor

def build_coatnet_5ch(num_classes=1, pretrained=False):
    """Build CoAtNet-Tiny adapted for 5-channel input."""
    model = timm.create_model(
        'coatnet_0_rw_224', pretrained=pretrained, num_classes=num_classes
    )
    original_conv = model.stem.conv1
    new_conv = nn.Conv2d(
        in_channels=5,
        out_channels=original_conv.out_channels,
        kernel_size=original_conv.kernel_size,
        stride=original_conv.stride,
        padding=original_conv.padding,
        bias=original_conv.bias is not None
    )
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = original_conv.weight.clone()
        nn.init.kaiming_normal_(
            new_conv.weight[:, 3:, :, :], mode='fan_out', nonlinearity='relu'
        )
        if original_conv.bias is not None:
            new_conv.bias.data = original_conv.bias.clone()
    model.stem.conv1 = new_conv
    return model


# Build model and load trained weights
coatnet = build_coatnet_5ch(num_classes=1, pretrained=False)
state_dict = torch.load(COATNET_WEIGHTS, map_location='cpu', weights_only=True)
coatnet.load_state_dict(state_dict)
coatnet = coatnet.to(device)
coatnet.eval()

# Remove classification head — we want penultimate features
# Hook into the global average pool output
coatnet_features = {}

def coatnet_hook(module, input, output):
    """Capture features before the classification head."""
    coatnet_features['embedding'] = output.detach()

# Register hook on the norm layer (just before head)
hook_handle = coatnet.head.global_pool.register_forward_hook(coatnet_hook)

# Test forward pass to get embedding dimension
with torch.no_grad():
    dummy = torch.zeros(1, 5, 224, 224, device=device)
    _ = coatnet(dummy)
    COATNET_DIM = coatnet_features['embedding'].shape[-1]

print(f'CoAtNet loaded successfully.')
print(f'  Embedding dim: {COATNET_DIM}')
print(f'  Parameters: {sum(p.numel() for p in coatnet.parameters()):,}')

CoAtNet loaded successfully.
  Embedding dim: 768
  Parameters: 26,667,907


In [ ]:
# 4.3 Load Whisper as frozen audio feature extractor

whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL)
whisper_model = WhisperModel.from_pretrained(WHISPER_MODEL).to(device)
whisper_model.eval()

print(f'Whisper loaded: {WHISPER_MODEL}')
print(f'  Encoder hidden dim: {WHISPER_DIM}')
print(f'  Parameters: {sum(p.numel() for p in whisper_model.parameters()):,}')

# Compute total fused feature dimension
# Video: mean + max + std of COATNET_DIM → 3 * COATNET_DIM
# Audio: WHISPER_DIM
VIDEO_FEAT_DIM = 3 * COATNET_DIM  # statistical pooling
AUDIO_FEAT_DIM = WHISPER_DIM
FUSED_DIM      = VIDEO_FEAT_DIM + AUDIO_FEAT_DIM

print(f'\nFused feature vector dimension:')
print(f'  Video (mean+max+std): {VIDEO_FEAT_DIM}')
print(f'  Audio (mean pool)  : {AUDIO_FEAT_DIM}')
print(f'  Total fused        : {FUSED_DIM}')

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Whisper loaded: openai/whisper-base
  Encoder hidden dim: 512
  Parameters: 72,593,920

Fused feature vector dimension:
  Video (mean+max+std): 2304
  Audio (mean pool)  : 512
  Total fused        : 2816


## 5. Video Frame Extraction & Processing

For each video:
1. Try to open the `.mp4` file and extract `NUM_FRAMES` uniformly sampled frames
2. **Fallback**: if the video is corrupted, look for frames in the `frames/` directory
3. If only 1 frame exists, replicate it to fill `NUM_FRAMES` slots (Batch Consistency)

In [ ]:
# 5.1 Frame extraction utilities (Safe Sequential Read)
import cv2
import numpy as np
import os

def extract_frames_from_video_safe(video_path, num_frames=5):
    """
    Extract frames sequentially to avoid seeking crashes.
    """
    frames = []
    cap = None
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return frames

        frame_buffer = []
        count = 0
        while True:
            ret, frame = cap.read()
            if not ret or count >= 300:
                break
            if frame is not None and frame.mean() > 5:
                frame_resized = cv2.resize(frame, (224, 224))
                frame_buffer.append(cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB))
            count += 1
            del frame

        if len(frame_buffer) > 0:
            indices = np.linspace(0, len(frame_buffer) - 1, num_frames, dtype=int)
            frames = [frame_buffer[i] for i in indices]
    except Exception:
        pass
    finally:
        if cap is not None:
            cap.release()
            del cap
    return frames

def extract_frames_fallback(video_path, meta_row, num_frames=5):
    frames_dir = os.path.join(
        FRAMES_ROOT, str(meta_row['type']), str(meta_row['race']),
        str(meta_row['gender']), str(meta_row['source'])
    )
    if not os.path.isdir(frames_dir): return []

    video_stem = os.path.splitext(str(meta_row['path']))[0]
    frame_files = sorted([f for f in os.listdir(frames_dir) if f.startswith(video_stem) and f.endswith('.jpg')])
    if not frame_files:
        frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])[:num_frames]

    frames = []
    for ff in frame_files[:num_frames]:
        img = cv2.imread(os.path.join(frames_dir, ff))
        if img is not None:
            img_resized = cv2.resize(img, (224, 224))
            frames.append(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
            del img, img_resized
    return frames

def get_video_frames(video_path, meta_row, num_frames=5):
    frames = extract_frames_from_video_safe(video_path, num_frames)
    if len(frames) == 0:
        frames = extract_frames_fallback(video_path, meta_row, num_frames)
    if len(frames) == 0:
        return None
    while len(frames) < num_frames:
        frames.append(frames[0].copy())
    return frames[:num_frames]

print('Frame extraction utilities loaded (Safe Sequential).')


Frame extraction utilities loaded (Safe Sequential).


## 6. Audio Feature Extraction

Extract audio from video using `librosa`, then pass through Whisper encoder.
The audio embedding is obtained via **mean pooling** over the encoder's temporal outputs.

In [ ]:
# 6.1 Audio extraction 
import uuid
import subprocess
import soundfile as sf
import torch
import os
import numpy as np

def extract_audio_embedding(video_path, processor, model, device):
    tmp_wav = f'/tmp/audio_{uuid.uuid4().hex[:8]}.wav'
    cmd = ['ffmpeg', '-y', '-v', 'error', '-i', video_path, '-t', '30', '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', tmp_wav]

    process = None
    try:
        # Use Popen instead of run so we can explicitly kill it on timeout
        process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        process.wait(timeout=5)  # 5 seconds max
    except subprocess.TimeoutExpired:
        # KILL THE ZOMBIE PROCESS!
        if process:
            process.kill()
            process.wait()
        return None
    except Exception:
        if process:
            process.kill()
            process.wait()
        return None

    try:
        if not os.path.exists(tmp_wav):
            return None

        audio_array, sr = sf.read(tmp_wav)
        if len(audio_array.shape) > 1:
            audio_array = audio_array.mean(axis=1)

        audio_array = audio_array.astype(np.float32)
        if len(audio_array) < 400:
            del audio_array
            return None

        inputs = processor(audio_array, sampling_rate=16000, return_tensors='pt')
        del audio_array

        input_features = inputs.input_features.to(device)
        del inputs

        with torch.no_grad():
            encoder_outputs = model.encoder(input_features)
            hidden_states = encoder_outputs.last_hidden_state
            audio_embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()

        del input_features, encoder_outputs, hidden_states
        return audio_embedding

    except Exception:
        return None
    finally:
        if os.path.exists(tmp_wav):
            try:
                os.remove(tmp_wav)
            except:
                pass

print('Audio extraction utilities loaded (Zombie Killer enabled).')


Audio extraction utilities loaded (Zombie Killer enabled).


In [ ]:
# 6.2 Video (CoAtNet) feature extraction (MEMORY-SAFE)

def extract_video_embedding(frames, coatnet_model, device):
    """
    Process NUM_FRAMES frames through CoAtNet + statistical pooling.
    Memory-safe: each frame tensor is deleted immediately after use.
    """
    frame_embeddings = []

    for frame_rgb in frames:
        tensor = prepare_5ch_tensor(frame_rgb, size=IMG_SIZE)
        tensor = tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                _ = coatnet_model(tensor)

        emb = coatnet_features['embedding'].squeeze().cpu().numpy()
        frame_embeddings.append(emb)

        # Explicit cleanup per frame
        del tensor, _
        coatnet_features.clear()

    frame_embeddings = np.array(frame_embeddings)

    # Statistical Pooling
    feat_mean = frame_embeddings.mean(axis=0)
    feat_max  = frame_embeddings.max(axis=0)
    feat_std  = frame_embeddings.std(axis=0)

    video_vector = np.concatenate([feat_mean, feat_max, feat_std])
    del frame_embeddings, feat_mean, feat_max, feat_std
    return video_vector


print('Video feature extraction loaded (memory-safe).')

Video feature extraction loaded (memory-safe).


## 7. Feature Extraction Pipeline — Full Dataset

Process every video in the dataset:
1. Extract frames → CoAtNet embeddings → Statistical Pooling → video vector
2. Extract audio → Whisper embedding → audio vector
3. Concatenate → fused vector
4. Save all features and labels to `.npy` files

In [ ]:
# 7.1 Two-Stage Feature Extraction (Video & Audio Separately)

import glob

MAX_TRAIN_SAMPLES = None  
VIDEO_DIM = 1280          # CoAtNet output dim
AUDIO_DIM = 512           # Whisper base dim

def extract_video_features(df, split_name='train'):
    limit = len(df) if MAX_TRAIN_SAMPLES is None else min(len(df), MAX_TRAIN_SAMPLES)
    if split_name == 'train' and MAX_TRAIN_SAMPLES:
        df = df.iloc[:limit].copy()

    print(f'\n[VIDEO] Extracting {split_name.upper()} features ({limit} samples)')
    chunk_size = 500
    chunk_features = []
    chunk_labels = []
    failed_count = 0

    for old_chunk in glob.glob(os.path.join(OUTPUT_DIR, f'{split_name}_video_chunk_*.npz')):
        os.remove(old_chunk)

    for idx in tqdm(range(limit), desc=f"{split_name} Video"):
        row = df.iloc[idx]
        video_path = row['abs_path'] if 'abs_path' in df.columns else os.path.join(DATASET_ROOT, str(row['type']), str(row['race']), str(row['gender']), str(row['source']), str(row['path']))
        label = row['label']

        try:
            frames = get_video_frames(video_path, row, NUM_FRAMES)
            if frames is None:
                video_vec = np.zeros(VIDEO_DIM, dtype=np.float32)
                failed_count += 1
            else:
                video_vec = extract_video_embedding(frames, coatnet, device)
                del frames
        except Exception:
            video_vec = np.zeros(VIDEO_DIM, dtype=np.float32)
            failed_count += 1

        chunk_features.append(video_vec)
        chunk_labels.append(label)

        if (idx + 1) % 50 == 0:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        if len(chunk_features) >= chunk_size:
            chunk_idx = idx // chunk_size
            np.savez(os.path.join(OUTPUT_DIR, f'{split_name}_video_chunk_{chunk_idx}.npz'),
                     X=np.array(chunk_features, dtype=np.float32), y=np.array(chunk_labels, dtype=np.int32))
            chunk_features.clear(); chunk_labels.clear()

    if len(chunk_features) > 0:
        np.savez(os.path.join(OUTPUT_DIR, f'{split_name}_video_chunk_final.npz'),
                 X=np.array(chunk_features, dtype=np.float32), y=np.array(chunk_labels, dtype=np.int32))

    all_X, all_y = [], []
    for cf in sorted(glob.glob(os.path.join(OUTPUT_DIR, f'{split_name}_video_chunk_*.npz'))):
        data = np.load(cf); all_X.append(data['X']); all_y.append(data['y']); os.remove(cf)

    X = np.concatenate(all_X, axis=0)
    y = np.concatenate(all_y, axis=0)
    print(f'Video extraction complete. Shape: {X.shape}, Failed: {failed_count}')
    np.save(os.path.join(OUTPUT_DIR, f'X_{split_name}_video.npy'), X)
    np.save(os.path.join(OUTPUT_DIR, f'y_{split_name}.npy'), y)
    return X, y


def extract_audio_features(df, split_name='train'):
    limit = len(df) if MAX_TRAIN_SAMPLES is None else min(len(df), MAX_TRAIN_SAMPLES)
    if split_name == 'train' and MAX_TRAIN_SAMPLES:
        df = df.iloc[:limit].copy()

    print(f'\n[AUDIO] Extracting {split_name.upper()} features ({limit} samples)')
    chunk_size = 500
    chunk_features = []
    failed_count = 0

    for old_chunk in glob.glob(os.path.join(OUTPUT_DIR, f'{split_name}_audio_chunk_*.npz')):
        os.remove(old_chunk)

    for idx in tqdm(range(limit), desc=f"{split_name} Audio"):
        row = df.iloc[idx]
        video_path = row['abs_path'] if 'abs_path' in df.columns else os.path.join(DATASET_ROOT, str(row['type']), str(row['race']), str(row['gender']), str(row['source']), str(row['path']))

        try:
            audio_vec = extract_audio_embedding(video_path, whisper_processor, whisper_model, device)
            if audio_vec is None:
                audio_vec = np.zeros(AUDIO_DIM, dtype=np.float32)
                failed_count += 1
        except Exception:
            audio_vec = np.zeros(AUDIO_DIM, dtype=np.float32)
            failed_count += 1

        chunk_features.append(audio_vec)

        if (idx + 1) % 50 == 0:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        if len(chunk_features) >= chunk_size:
            chunk_idx = idx // chunk_size
            np.savez(os.path.join(OUTPUT_DIR, f'{split_name}_audio_chunk_{chunk_idx}.npz'),
                     X=np.array(chunk_features, dtype=np.float32))
            chunk_features.clear()

    if len(chunk_features) > 0:
        np.savez(os.path.join(OUTPUT_DIR, f'{split_name}_audio_chunk_final.npz'),
                 X=np.array(chunk_features, dtype=np.float32))

    all_X = []
    for cf in sorted(glob.glob(os.path.join(OUTPUT_DIR, f'{split_name}_audio_chunk_*.npz'))):
        data = np.load(cf); all_X.append(data['X']); os.remove(cf)

    X = np.concatenate(all_X, axis=0)
    print(f'Audio extraction complete. Shape: {X.shape}, Failed: {failed_count}')
    np.save(os.path.join(OUTPUT_DIR, f'X_{split_name}_audio.npy'), X)
    return X


In [ ]:
# 7.2 Execute Split Extraction and Hardened Fusion
import os
import numpy as np

# Stage 1: Video Extraction
print("Starting Stage 1: VIDEO EXTRACTION")
extract_video_features(train_df, split_name='train')
extract_video_features(test_df, split_name='test')

# Stage 2: Audio Extraction
print("\nStarting Stage 2: AUDIO EXTRACTION")
extract_audio_features(train_df, split_name='train')
extract_audio_features(test_df, split_name='test')

# 7.3 Hardened Feature Fusion (Loading from Disk)
print("\nStarting Stage 3: FEATURE FUSION (Disk-Based)")

video_train_path = os.path.join(OUTPUT_DIR, 'X_train_video.npy')
audio_train_path = os.path.join(OUTPUT_DIR, 'X_train_audio.npy')
video_test_path  = os.path.join(OUTPUT_DIR, 'X_test_video.npy')
audio_test_path  = os.path.join(OUTPUT_DIR, 'X_test_audio.npy')

# Verify files exist to ensure safe fusion
for p in [video_train_path, audio_train_path, video_test_path, audio_test_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required feature file: {p}")

print("Loading Video Features...")
X_train_video = np.load(video_train_path)
X_test_video  = np.load(video_test_path)

print("Loading Audio Features...")
X_train_audio = np.load(audio_train_path)
X_test_audio  = np.load(audio_test_path)

# Labels are saved during Video Extraction
y_train = np.load(os.path.join(OUTPUT_DIR, 'y_train.npy'))
y_test  = np.load(os.path.join(OUTPUT_DIR, 'y_test.npy'))

print("Concatenating Features...")
X_train_fused = np.concatenate([X_train_video, X_train_audio], axis=1)
X_test_fused  = np.concatenate([X_test_video, X_test_audio], axis=1)

print(f"Final Fused Train Shape: {X_train_fused.shape}")
print(f"Final Fused Test Shape : {X_test_fused.shape}")

np.save(os.path.join(OUTPUT_DIR, 'X_train_fused.npy'), X_train_fused)
np.save(os.path.join(OUTPUT_DIR, 'X_test_fused.npy'), X_test_fused)
print("Fusion complete and secured to disk.")

Starting Stage 1: VIDEO EXTRACTION

[VIDEO] Extracting TRAIN features (17252 samples)


train Video: 100%|██████████| 17252/17252 [53:46<00:00,  5.35it/s]


Video extraction complete. Shape: (17252, 2304), Failed: 0

[VIDEO] Extracting TEST features (4314 samples)


test Video: 100%|██████████| 4314/4314 [13:11<00:00,  5.45it/s]


Video extraction complete. Shape: (4314, 2304), Failed: 0

Starting Stage 2: AUDIO EXTRACTION

[AUDIO] Extracting TRAIN features (17252 samples)


train Audio: 100%|██████████| 17252/17252 [58:55<00:00,  4.88it/s]


Audio extraction complete. Shape: (17252, 512), Failed: 0

[AUDIO] Extracting TEST features (4314 samples)


test Audio: 100%|██████████| 4314/4314 [14:47<00:00,  4.86it/s]


Audio extraction complete. Shape: (4314, 512), Failed: 0

Starting Stage 3: FEATURE FUSION (Disk-Based)
Loading Video Features...
Loading Audio Features...
Concatenating Features...
Final Fused Train Shape: (17252, 2816)
Final Fused Test Shape : (4314, 2816)
Fusion complete and secured to disk.


## 8. Release GPU Resources

Free CoAtNet and Whisper from GPU — we only need CPU for XGBoost.

In [ ]:
# 8. Free GPU memory — XGBoost runs on CPU
hook_handle.remove()
del coatnet, whisper_model, whisper_processor
gc.collect()
torch.cuda.empty_cache()
print('GPU memory released. Proceeding with XGBoost on CPU.')

GPU memory released. Proceeding with XGBoost on CPU.


## 9. XGBoost Fusion Classifier

Train XGBoost on the concatenated (video + audio) feature vectors.
- Handles class imbalance via `scale_pos_weight`
- Early stopping on AUC
- Saves the trained model for production deployment

In [ ]:
# 9.1 Train XGBoost

# Handle class imbalance
n_bonafide = (y_train == 0).sum()
n_spoof    = (y_train == 1).sum()
scale_weight = n_bonafide / n_spoof if n_spoof > n_bonafide else n_spoof / n_bonafide

print(f'Training XGBoost Fusion Classifier...')
print(f'  Train samples  : {len(y_train)}')
print(f'  Feature dim    : {X_train_fused.shape[1]}')
print(f'  Bonafide/Spoof : {n_bonafide}/{n_spoof}')
print(f'  scale_pos_weight: {scale_weight:.4f}')

xgb_clf = xgb.XGBClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=7,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_weight,
    random_state=RANDOM_SEED,
    eval_metric='auc',
    early_stopping_rounds=50,
    tree_method='hist',  # fast histogram-based method
    n_jobs=-1
)

xgb_clf.fit(
    X_train_fused, y_train,
    eval_set=[(X_train_fused, y_train), (X_test_fused, y_test)],
    verbose=50
)

print('\nXGBoost training complete.')

Training XGBoost Fusion Classifier...
  Train samples  : 17252
  Feature dim    : 2816
  Bonafide/Spoof : 400/16852
  scale_pos_weight: 0.0237
[0]	validation_0-auc:0.85312	validation_1-auc:0.74003
[50]	validation_0-auc:0.99210	validation_1-auc:0.81361
[61]	validation_0-auc:0.99505	validation_1-auc:0.81650

XGBoost training complete.


In [ ]:
# .1 Save XGBoost model

model_path = os.path.join(OUTPUT_DIR, 'xgboost_fusion_model.joblib')
joblib.dump(xgb_clf, model_path)
print(f'XGBoost model saved: {model_path}')

# Also save in XGBoost native format
xgb_native_path = os.path.join(OUTPUT_DIR, 'xgboost_fusion_model.json')
xgb_clf.save_model(xgb_native_path)
print(f'XGBoost native model saved: {xgb_native_path}')

# .2 Save feature extraction config for inference

inference_config = {
    'pipeline': 'Intermediate Fusion (CoAtNet + Whisper + XGBoost)',
    'coatnet_model': 'coatnet_0_rw_224',
    'coatnet_input_channels': 5,
    'coatnet_input_size': IMG_SIZE,
    'coatnet_weights': 'best_coatnet_5ch.pth',
    'whisper_model': WHISPER_MODEL,
    'num_frames': NUM_FRAMES,
    'frame_sampling': 'uniform',
    'video_pooling': 'statistical (mean + max + std)',
    'audio_pooling': 'mean over time',
    'video_feat_dim': int(VIDEO_FEAT_DIM),
    'audio_feat_dim': int(AUDIO_FEAT_DIM),
    'fused_feat_dim': int(FUSED_DIM),
    'classifier': 'XGBoost',
    'xgboost_model_file': 'xgboost_fusion_model.joblib',
    'label_mapping': {'0': 'Bonafide (Real)', '1': 'Spoof (Fake)'},
    'normalization': {
        'rgb_mean': [0.485, 0.456, 0.406],
        'rgb_std': [0.229, 0.224, 0.225],
        'forensic_mean': [0.5, 0.5],
        'forensic_std': [0.5, 0.5]
    }
}

with open(os.path.join(OUTPUT_DIR, 'inference_config.json'), 'w') as f:
    json.dump(inference_config, f, indent=4)

print(f'Inference config saved.')

# .3 Save predictions for downstream analysis

np.savez(
    os.path.join(OUTPUT_DIR, 'test_predictions.npz'),
    y_true=y_test,
    y_pred=y_pred,
    y_proba=y_proba
)

print(f'Test predictions saved.')

# .4 Summary of all saved artifacts

print(f'\n{"="*60}')
print('SAVED ARTIFACTS SUMMARY')
print(f'{"="*60}')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<45s} {size_mb:>8.2f} MB')

print(f'\n Pipeline complete! All artifacts saved to: {OUTPUT_DIR}')